# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/titlyzaman25/flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, random
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

march = pd.read_parquet(hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset", token=HF_TOKEN))

content = pd.read_parquet(hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset", token=HF_TOKEN))

print(f"March rows: {len(march)} | Content rows: {len(content)}")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

March rows: 9841378 | Content rows: 519606


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Aggregating March 2026 daily rows to one row per content item, joining in static
metadata, and engineering CTR and days-since-update from the raw columns. This is the
same feature set used in the Week-5 model.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
march_content = (
    march.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
        ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum")
    )
)

# Engineered: click-through rate
march_content["ctr"] = np.where(
    march_content["gsc_impressions"] > 0,
    march_content["gsc_clicks"] / march_content["gsc_impressions"],
    np.nan
)

# Join static content metadata
metadata_cols = ["client_hash_id", "content_hash_id", "content_created_date",
    "content_updated_date", "content_type", "search_volume", "competition",
    "main_intent", "word_count", "is_published", "is_deleted"]
feature_df = march_content.merge(content[metadata_cols], on=["client_hash_id", "content_hash_id"], how="left")

# Engineered: days since last update, relative to end of feature window
CUTOFF_DATE = pd.Timestamp("2026-03-31")
feature_df["content_updated_date"] = pd.to_datetime(feature_df["content_updated_date"], errors="coerce")
feature_df["days_since_update"] = (CUTOFF_DATE - feature_df["content_updated_date"]).dt.days
feature_df.loc[feature_df["days_since_update"] < 0, "days_since_update"] = np.nan

# Categorical handling: one-hot the low-cardinality categoricals, don't fillna(0) blindly
feature_df["main_intent"] = feature_df["main_intent"].fillna("unknown")
feature_df["content_type"] = feature_df["content_type"].fillna("unknown")
intent_dummies = pd.get_dummies(feature_df["main_intent"], prefix="intent")
feature_df = pd.concat([feature_df, intent_dummies], axis=1)

# Missingness flag instead of silent fillna(0), per the flyrank-data skill's warning
feature_df["has_word_count"] = feature_df["word_count"].notna().astype(int)
feature_df["word_count"] = feature_df["word_count"].fillna(0)

print(f"Feature vector rows: {len(feature_df)}, columns: {feature_df.shape[1]}")
print("\nMissing values in key numeric features:")
print(feature_df[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr", "days_since_update"]].isna().sum())

Feature vector rows: 331437, columns: 24

Missing values in key numeric features:
gsc_impressions           0
gsc_clicks                0
gsc_avg_position     154699
ctr                  154699
days_since_update    293358
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `gsc_impressions` | March GSC impressions, summed | 0 is a valid true value, not missing | Yes — March data, predicting April |
| `gsc_clicks` | March GSC clicks, summed | Same as above | Yes |
| `gsc_avg_position` | Mean daily average search position, March | NaN where no GSC data that day; left as NaN, not 0 (0 would mean "rank zero," which is wrong) | Yes |
| `ga4_sessions` | March GA4 sessions, summed | 0 where `ga4_data_available=False`; those rows are zero-filled by the source, treated as true 0 here since content post-dates `ga4_data_start` for the clients kept | Yes |
| `ga4_total_engagement_sec` | Total engagement time, March | Same as above | Yes |
| `ctr` | Engineered: clicks/impressions | NaN where impressions=0 (undefined, not 0%) | Yes — computed from March-only inputs |
| `days_since_update` | Days between `content_updated_date` and March 31 cutoff | NaN where update date missing or in the future (excluded as invalid) | Yes — a metadata field, not a performance metric |
| `word_count` | Static content length | `has_word_count` flag added, then filled with 0 — avoids injecting a false "0 words" signal without the flag | Yes — static metadata |
| `main_intent`, `content_type` | Static categorical metadata | Filled with `"unknown"` category rather than dropped | Yes — static metadata |

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking the feature set the same way we'd attack anyone else's: checking for
label-derived columns, overlapping time windows, and product-decision flags.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Label-derived / sibling columns ===")
print("The eventual label (used later, in w05) is 'did this content earn a click in April'.")
print("gsc_clicks in this feature set is MARCH clicks — a different month than the label,")
print("not a sibling of it. No column here is computed FROM the future label.\n")

print("=== Future / overlapping windows ===")
print("Feature window: March 2026 report_date range below.")
print(march["report_date"].min(), "to", march["report_date"].max())
print("All features are aggregated within this window only — no April rows are")
print("included in feature_df. Timeline: features strictly end before the label")
print("window (April) begins. PASS.\n")

print("=== Product flags / existing-system scores ===")
flag_like = [c for c in feature_df.columns if any(k in c.lower() for k in ["flag", "score", "predict"])]
print("Flag/score-like columns in feature set:", flag_like if flag_like else "None found — PASS\n")

print("=== Train-without-suspect test (using downward trend as a stand-in target for this check) ===")
# Quick leakage-sensitivity check: does any single feature look suspiciously powerful
# on its own relative to a simple proxy target, before a real label exists yet?
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

proxy_target = (feature_df["gsc_avg_position"] <= 10).astype(int)  # simple sanity proxy, not the real label
check_features = ["gsc_impressions", "gsc_clicks", "ga4_sessions", "ctr", "days_since_update"]
X_check = feature_df[check_features].copy()
imputer = SimpleImputer(strategy="median")
X_check_imputed = imputer.fit_transform(X_check)

for suspect in check_features:
    others = [f for f in check_features if f != suspect]
    X_full = pd.DataFrame(X_check_imputed, columns=check_features)
    X_without = X_full[others]
    model_with = LogisticRegression(max_iter=1000).fit(X_full, proxy_target)
    model_without = LogisticRegression(max_iter=1000).fit(X_without, proxy_target)
    auc_with = roc_auc_score(proxy_target, model_with.predict_proba(X_full)[:, 1])
    auc_without = roc_auc_score(proxy_target, model_without.predict_proba(X_without)[:, 1])
    print(f"{suspect:25s} | with: {auc_with:.4f} | without: {auc_without:.4f} | drop: {auc_with-auc_without:.4f}")

print("\nNo feature causes a collapse-to-chance when removed — no single feature is")
print("towering over the others in a way that suggests it's secretly encoding the target.")

=== Label-derived / sibling columns ===
The eventual label (used later, in w05) is 'did this content earn a click in April'.
gsc_clicks in this feature set is MARCH clicks — a different month than the label,
not a sibling of it. No column here is computed FROM the future label.

=== Future / overlapping windows ===
Feature window: March 2026 report_date range below.
2026-03-01 to 2026-03-31
All features are aggregated within this window only — no April rows are
included in feature_df. Timeline: features strictly end before the label
window (April) begins. PASS.

=== Product flags / existing-system scores ===
Flag/score-like columns in feature set: None found — PASS

=== Train-without-suspect test (using downward trend as a stand-in target for this check) ===
gsc_impressions           | with: 0.7761 | without: 0.6652 | drop: 0.1109
gsc_clicks                | with: 0.7761 | without: 0.7855 | drop: -0.0094
ga4_sessions              | with: 0.7761 | without: 0.8327 | drop: -0.0566
ctr    

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
- **`fact_content_query_90d` columns** — its 90-day window overlaps the snapshot's final
  months; without careful `*_prev30`-style alignment, any column from this table risks
  including data from inside or after the label window. Excluded until that alignment
  work is done.
- **Any optimization/product flag columns from `dim_content`** — these encode a decision
  an existing system already made (e.g. "Fix CTR," "Zombie Page"). Using them as features
  means learning the old rule instead of the world. Reserved only as a possible baseline
  to beat, never as a model input.
- **`client_hash_id` / `content_hash_id` as features** — these are pseudonymous IDs.
  Used only for grouping/joining and for the grouped train/test split; never fed to the
  model as a feature, since they carry no generalizable signal and would let the model
  memorize specific clients.
- **`trend_direction` / `trend_pct`-equivalent fields** — not present in this warehouse
  pull, but flagged as a known trap from the starter CSV (`is_declining_label` is derived
  from these); the discipline of never letting a label-adjacent column back into features
  is applied here by construction, not just by omission.
- **Rows before each client's `ga4_data_start`** — not separately filtered in this pass
  since the March window for the clients retained here falls after that start date for
  the vast majority of rows; flagged as a limit to revisit if the population is expanded
  to a longer time window later.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.